## 1. Environment and Drive setup

Set `REPO_DIR` to your Stage 2 checkout. On Colab this cell mounts Drive and clones the feature branch if the checkout is missing. The remaining cells also run in a local Jupyter kernel.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
REPO_DIR = Path(os.environ.get("SANVIA_STAGE2_REPO", "/content/drive/MyDrive/SANVIA" if COLAB else Path.cwd()))
if REPO_DIR.name == "notebooks":
    REPO_DIR = REPO_DIR.parent
if COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", "stage2-domain-adaptation", "https://github.com/yanal-alshoufi/Sanvia-AI-for-Early-Hope.git", str(REPO_DIR)], check=True)
if not (REPO_DIR / "configs/stage2/default.yaml").is_file():
    raise FileNotFoundError(f"Set REPO_DIR to the Stage 2 checkout. Current path: {REPO_DIR}")
if COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[stage2,dev]"], check=True)
import pandas as pd, torch, yaml
from IPython.display import display, Image
from sanvia.utils.io import load_json
print(sys.version, torch.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU")

## 2. Verify the repository

Inspect the revision and working tree before starting a research run.

In [ ]:
from sanvia.mlops.git_metadata import get_git_metadata
git = get_git_metadata(REPO_DIR)
display({k: git[k] for k in ("branch", "commit", "dirty")})

## 3. Configure Stage 2

The default smoke run generates deterministic fixtures and uses two designated fixture backbones. For a research run, set `MODE` to `full`, choose an authorized output directory and supply the source versions and annotation entries in a private YAML override. `pilot` also requires verified inputs. The primary comparison uses both strategies.

In [ ]:
from sanvia.stage2.cli import smoke_config
from sanvia.stage2.pipeline import Benchmark, configuration
from sanvia.utils.config import deep_update
MODE = "smoke"
STAGE1_OUTPUT_DIR = Path("/content/drive/MyDrive/VnDir_Mammo/sanvia_outputs")
VINDR_IMAGES = Path("/content/drive/MyDrive/VnDir_Mammo/images/images_png")
DATASET_ROOTS = {s: Path("/content/drive/MyDrive/SANVIA_Datasets") / s for s in ("CBIS-DDSM","CMMD","INbreast")}
OUTPUT_DIR = Path(os.environ.get("SANVIA_STAGE2_OUTPUT", "/content/drive/MyDrive/SANVIA_Stage2_Outputs" if COLAB else REPO_DIR / "outputs/stage2"))
PRETRAINED = MODE != "smoke"
RESUME = True
ENABLED_SOURCES = list(DATASET_ROOTS)
SELECTED_STRATEGIES = ["sequential","multi-dataset"]
SEEDS = [42] if MODE != "full" else [42,123,2026]
OVERRIDES = {}  # Or yaml.safe_load(Path("/private/stage2.yaml").read_text())
overrides = smoke_config(OUTPUT_DIR) if MODE == "smoke" else {}
deep_update(overrides, dict(mode=MODE, output=str(OUTPUT_DIR), stage1_output=str(STAGE1_OUTPUT_DIR), vindr_images=str(VINDR_IMAGES),
                            sources={k: {"root": str(v)} for k,v in DATASET_ROOTS.items()}, pretrained=PRETRAINED, enabled_sources=ENABLED_SOURCES,
                            strategies=SELECTED_STRATEGIES, seeds=SEEDS))
cfg = configuration(REPO_DIR, deep_update(overrides, OVERRIDES))
display({k:cfg[k] for k in ("name","mode","enabled_sources","pretrained","seeds","output")})

## 4. Load the Stage 1 artifacts

Real modes verify the saved study assignments, eligible images and preprocessing. The smoke fixture has its own explicitly designated pair.

In [ ]:
bench = Benchmark(REPO_DIR, cfg)
cfg = bench.cfg
snap = bench.snapshot
display(dict(backbones=snap.selected, dataset=snap.data.dataset_id, split=snap.data.split_id, age=snap.data.age_enabled,
             validation_breasts=len(snap.data.val_df), compatibility={k:v["incompatibilities"] for k,v in snap.baselines.items()}))

## 5. Verify dataset paths

Check the roots, annotation files and free output space before indexing.

In [ ]:
import shutil
from sanvia.stage2.adapters import ADAPTERS
for s in cfg["enabled_sources"]:
    adapter = ADAPTERS[s](cfg["sources"][s])
    for entry in adapter.annotations:
        if not adapter.annotation_path(entry).is_file():
            raise FileNotFoundError(adapter.annotation_path(entry))
    print(s, adapter.root, cfg["sources"][s]["version"])
print("Output space (GiB)", round(shutil.disk_usage(bench.out).free / 2**30, 1))

## 6. Index and version the sources

Indexing, normalization and the audit share one pass through the source metadata. Prepared assignments and accepted manifest fingerprints persist under the benchmark identity.

In [ ]:
audit_dir = bench.execute("prepare")
display(pd.DataFrame([dict(dataset=s, version=d["version"], fingerprint=d["fingerprint"], split=d["split_id"]) for s,d in bench.data.items()]))

## 7. Review the audit

Inspect absolute counts, label issues, exclusions and cross-source fingerprint matches. An unresolved critical issue blocks training.

In [ ]:
for s in bench.data:
    display(s, load_json(audit_dir/s/"audit_summary.json"), Image(filename=str(audit_dir/s/"class_distributions.png")))
display(load_json(audit_dir/"critical_issues.json"), pd.read_csv(audit_dir/"duplicate_candidates.csv"))

## 8. Inspect task definitions

Check native vocabularies, scope, mappings and masks. Positive-only findings retain unknown labels for unannotated classes.

In [ ]:
for s,d in bench.data.items():
    display(s, pd.DataFrame(d["tasks"]).T)
    labels = pd.read_csv(audit_dir/s/"task_labels.csv")
    display(labels.groupby(["task","scope","valid"],dropna=False).size().rename("instances").to_frame())

## 9. Check source partitions

Review patient-group counts and inspect the persisted assignments before training.

In [ ]:
for s in bench.data:
    splits = pd.read_csv(audit_dir/s/"split_assignments.csv")
    assert splits.patient_id.is_unique
    display(s, splits.groupby("final_split").size().rename("patients").to_frame())

## 10. Review preprocessing

Set `preview_images: true` in your private overrides to save authorized examples. All source formats use the backbone resolution and aspect-preserving padding.

In [ ]:
from sanvia.models.backbone_factory import get_backbone_spec
spec = get_backbone_spec(snap.config,snap.selected[0])
for s in bench.data:
    preview = audit_dir/s/"preprocessing.png"
    if preview.is_file(): display(Image(filename=str(preview)))
print("Backbone input resolution",spec.input_size)

## 11. Inspect a source batch

Check tensor dimensions and valid slots on a deterministic validation batch.

In [ ]:
from sanvia.stage2.source_data import SourceDataset, loader
source = cfg["enabled_sources"][0]
dataset = SourceDataset(bench.data[source],Path(cfg["sources"][source]["root"]),"validation",spec.input_size,
                        snap.config["data"]["preprocessing"],snap.config["data"]["augmentation"],None)
batch = next(iter(loader(dataset,bench.settings(spec.name))))
display({k:list(v.shape) for k,v in batch.items() if torch.is_tensor(v)})
display(batch["view_mask"],{k:v for k,v in batch.items() if k.endswith("_mask") and k!="view_mask"})

## 12. Initialize the source models

Verify the encoder feature size and routed output dimensions for each selected backbone.

In [ ]:
from sanvia.models.backbone_factory import build_backbone, infer_feature_dim
from sanvia.stage2.source_model import SourceModel
registry = {s:d["tasks"] for s,d in bench.data.items()}
for name in snap.selected:
    sp = get_backbone_spec(snap.config,name)
    m = SourceModel(build_backbone(sp,pretrained=cfg["pretrained"]),registry).eval()
    assert infer_feature_dim(m.backbone)==sp.feature_dim
    with torch.no_grad(): logits=m(torch.zeros(1,2,3,sp.input_size,sp.input_size),torch.tensor([[True,False]]),source)
    display(name, sum(p.numel() for p in m.parameters()), {k:list(v.shape) for k,v in logits.items()})
    del m

## 13. Initialize tracking

Use one writer per output root. MLflow and the optional DVC hash references have separate Stage 2 storage.

In [ ]:
from sanvia.stage2.tracking import paths_for, EXPERIMENT
from sanvia.mlops.mlflow_tracking import setup_mlflow, verify_mlflow_persistence
if cfg["mlflow"]:
    with bench.lock:
        tracking_cfg, paths = paths_for(bench.out)
        setup_mlflow(tracking_cfg, paths)
        display(verify_mlflow_persistence(paths, EXPERIMENT))
display(dict(mlflow=cfg["mlflow"], dvc=load_json(bench.out/"dvc_status.json") if cfg["dvc"] else "disabled"))

## 14. Run preflight tests

Run the complete synthetic regression suite before starting the experiment cells.

In [ ]:
from sanvia.stage2.preflight import preflight
display(preflight(REPO_DIR,bench.out,bench.code))

## 15. Inspect the experiment matrix

The split, source order and per-source quotas define the comparison. CBIS → CMMD → INbreast is a predefined hypothesis.

In [ ]:
display(pd.DataFrame(bench.plan()))
from sanvia.stage2.source_training import plan_sources
from collections import Counter
for strategy in cfg["strategies"]:
    for stage,updates,seen in plan_sources(cfg["enabled_sources"],bench.settings(spec.name),strategy,cfg["seeds"][0]):
        print(strategy,stage,Counter((s,"warmup" if frozen else "train") for s,frozen,replay in updates))

## 16. Sequential supervised adaptation

Train each source from the preceding selected source checkpoint, preserving source heads for forgetting evaluation.

In [ ]:
if "sequential" in cfg["strategies"]: display(bench.execute("train-sequential",RESUME))

## 17. Multi-dataset supervised adaptation

Train homogeneous source batches with a shared encoder and source-specific heads.

In [ ]:
if "multi-dataset" in cfg["strategies"]: display(bench.execute("train-multi-dataset",RESUME))

## 18. Inspect source validation and forgetting

Each matrix uses held-out source patients and the selected checkpoint at each stage.

In [ ]:
for matrix in sorted(bench.out.glob("*/*/*/source/forgetting_matrix.csv")):
    display(str(matrix.relative_to(bench.out)),pd.read_csv(matrix,index_col=0),Image(filename=str(matrix.with_name("forgetting.png"))))

## 19. Verify source-only backbone exports

Reload each export using the recorded backbone specification and checksum.

In [ ]:
from sanvia.stage2.export import load_export
for path in sorted(bench.out.glob("*/*/*/source/source_adapted_backbone.pt")):
    encoder=load_export(path)
    print(path.relative_to(bench.out), encoder.feature_dim)
    del encoder

## 20. Controlled VinDr fine-tuning

The same Stage 1 factory, target-specific initialization, split, transforms, losses and training settings apply to all three arms for a seed.

In [ ]:
display(bench.execute("finetune-vindr",RESUME))

## 21. Review target validation

Inspect the validation learning curves, confusion matrices, probabilities and class support in private storage.

In [ ]:
for path in sorted(bench.out.glob("*/*/*/target/target_summary.json")):
    r=load_json(path)
    display(r["backbone"],r["seed"],r["strategy"],Image(filename=str(path.with_name("learning_curves.png"))))
    for task in ("birads","density"):
        display(pd.read_csv(path.with_name("per_class_"+task+".csv"),index_col=0),Image(filename=str(path.with_name("confusion_"+task+".png"))))

## 22. Compare against matched baselines

Every difference is paired by backbone, target seed, saved split and target protocol.

In [ ]:
complete_matrix = set(cfg["strategies"]) == {"sequential","multi-dataset"}
comparison = bench.out/"comparison"
if complete_matrix:
    bench.execute("compare", RESUME)
    display(pd.read_csv(comparison/"seed_results.csv"), Image(filename=str(comparison/"delta.png")))
else:
    print("Both strategies are required for the primary comparison.")

## 23. Select initializations independently

The selection report records per-task regression guards, seed stability and the predefined tie break. Smoke and pilot decisions are ineligible for research handoff.

In [ ]:
selection = load_json(comparison/"selection_report.json") if complete_matrix else {"eligible_for_handoff": False, "decisions": {}}
display(selection)

## 24. Registry and Stage 3 handoff

A completed full benchmark exports the chosen pre-target weights and registers research candidates if MLflow is enabled.

In [ ]:
if selection["eligible_for_handoff"]:
    bench.execute("export",RESUME)
    display(load_json(bench.out/"stage3_handoff/selected_initializations.json"))
else:
    print("Research handoff requires full mode.")

## 25. Review reproducibility

The local report records source identities, target identity and split, environment, code digest and eligible baseline evidence.

In [ ]:
report=load_json(bench.out/"reproducibility_report.json")
display({k: report[k] for k in ("mode","experiment","git","stage1_commit","target_dataset","target_split","sources","environment","official_test_used")})

## 26. Summarize the benchmark

Use the private output directory for validation-level records and all medical images.

In [ ]:
target_runs=[load_json(p) for p in sorted(bench.out.glob("*/*/*/target/target_summary.json"))]
results=pd.DataFrame([{**{k:r[k] for k in ("backbone","seed","strategy","training_time_s")}, "score":r["validation"]["selection_score"]} for r in target_runs])
display(results)
display(dict(mode=bench.mode, selected=selection["decisions"], completed=len(results), enabled_sources=cfg["enabled_sources"],
             output=str(bench.out), source_exports=[str(p.relative_to(bench.out)) for p in bench.out.glob("*/*/*/source/source_adapted_backbone.pt")],
             handoff=str(bench.out/"stage3_handoff") if selection["eligible_for_handoff"] else None,
             tracking_failures=[str(p.relative_to(bench.out)) for p in bench.out.rglob("mlflow_failure.json")],
             official_vindr_test_used=report["official_test_used"]))